# Milestone audit: split statistics, privacy scan, and M3 result uncertainty

**Purpose.** This closes three evidence gaps found while auditing M1–M3 against
`docs/milestone-framework.md`:

1. **M2 statistical analysis.** Formal split comparisons, beyond Notebook 02's descriptive counts.
2. **M2 privacy assessment.** A measured scan of the report text for personal-data patterns.
3. **M3 result uncertainty.** Confidence intervals and paired significance tests for the dev comparisons.

**Leakage rules.** Input-side variables (question, table, text) are compared across all three splits,
because they are model inputs and not labels. Label-side variables (answer type, program structure) are
compared for **train vs dev only**; no new statistic is computed on test labels. The privacy scan reads
only report text, never answers or programs. M3 uncertainty uses the saved dev prediction artifacts.
Seed 42 is used for all resampling.

In [1]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from finevidence.data import load_finqa_split, parse_finqa_program
from finevidence.data.acquisition import PINNED_SHA256, file_sha256
from finevidence.evaluation import compare_execution
from finevidence.reasoning import official_str_to_num

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SEED = 42
for split in ("train", "dev", "test"):
    assert file_sha256(RAW_DIR / f"{split}.json") == PINNED_SHA256[split], split
splits = {name: load_finqa_split(RAW_DIR / f"{name}.json") for name in ("train", "dev", "test")}
print({name: len(records) for name, records in splits.items()}, "| checksums verified")

{'train': 6251, 'dev': 883, 'test': 1147} | checksums verified


## 1. Input-side split comparison (all splits)

Two-sample Kolmogorov–Smirnov tests compare each split's distribution with train's. With thousands of
records, tiny differences can be statistically significant, so the KS statistic *D* (the largest gap
between the two cumulative distributions, from 0 to 1) is the effect size to read. *D* below 0.05 means
the distributions are practically identical.

In [2]:
NUMERIC_CELL = re.compile(r"\d")


def input_features(record) -> dict[str, float]:
    table = record["table"]
    cells = [str(cell) for row in table for cell in row]
    text = record["pre_text"] + record["post_text"]
    return {
        "question_words": len(record["qa"]["question"].split()),
        "table_rows": len(table),
        "table_columns": len(table[0]) if table else 0,
        "text_sentences": len(text),
        "context_words": sum(len(s.split()) for s in text) + sum(len(c.split()) for c in cells),
        "numeric_cell_share": sum(bool(NUMERIC_CELL.search(c)) for c in cells) / max(1, len(cells)),
        "question_mentions_year": float(
            bool(re.search(r"\b(?:19|20)\d{2}\b", record["qa"]["question"]))
        ),
    }


features = {
    name: pd.DataFrame([input_features(r) for r in records]) for name, records in splits.items()
}
rows = []
for column in features["train"].columns:
    row = {"feature": column, "train_median": features["train"][column].median()}
    for other in ("dev", "test"):
        result = stats.ks_2samp(features["train"][column], features[other][column])
        row[f"{other}_median"] = features[other][column].median()
        row[f"KS_D_train_vs_{other}"] = round(result.statistic, 4)
        row[f"p_train_vs_{other}"] = round(result.pvalue, 4)
    rows.append(row)
display(pd.DataFrame(rows))
print(
    "train share of questions mentioning a year:",
    round(features["train"]["question_mentions_year"].mean(), 4),
)

,feature,train_median,dev_median,KS_D_train_vs_dev,p_train_vs_dev,test_median,KS_D_train_vs_test,p_train_vs_test
0,question_words,15.00,15.00,0.0253,0.6941,16.00,0.0359,0.1607
1,table_rows,6.00,6.00,0.0198,0.9158,6.00,0.0766,0.0000
2,table_columns,4.00,4.00,0.0269,0.6196,4.00,0.0187,0.8805
3,text_sentences,23.00,24.00,0.0753,0.0003,23.00,0.0419,0.0643
4,context_words,681.00,690.00,0.0510,0.0341,671.00,0.0417,0.0664
5,numeric_cell_share,0.75,0.75,0.0716,0.0007,0.75,0.0415,0.0692
6,question_mentions_year,1.00,1.00,0.0232,0.7909,1.00,0.0005,1.0000


train share of questions mentioning a year: 0.7695


## 2. Label-side comparison (train vs dev only)

Chi-square tests of homogeneity on answer type, program length and operation family. Cramér's *V*
(0 = identical proportions, 1 = completely different) is the effect size. Test labels are excluded.

In [3]:
def answer_type(answer: str) -> str:
    answer = answer.strip()
    if not answer:
        return "empty"
    if answer.lower() in {"yes", "no"}:
        return "yes/no"
    if re.fullmatch(r"-?[\d.,]+%", answer):
        return "percent"
    if re.fullmatch(r"-?[\d.,]+", answer):
        return "number"
    return "other"


def label_features(record) -> dict[str, str]:
    steps = parse_finqa_program(record["qa"]["program"]).steps
    operations = {step.operation for step in steps}
    family = (
        "table_op"
        if any(op.startswith("table_") for op in operations)
        else "comparison"
        if "greater" in operations
        else "arithmetic"
    )
    return {
        "answer_type": answer_type(record["qa"]["answer"]),
        "program_steps": "1" if len(steps) == 1 else "2" if len(steps) == 2 else "3+",
        "operation_family": family,
        "gold_facts": "1" if len(record["qa"]["gold_inds"]) == 1 else "2+",
    }


labels = {
    name: pd.DataFrame([label_features(r) for r in splits[name]]) for name in ("train", "dev")
}
rows = []
for column in labels["train"].columns:
    table = pd.crosstab(
        pd.concat([labels["train"][column], labels["dev"][column]]),
        ["train"] * len(labels["train"]) + ["dev"] * len(labels["dev"]),
    )
    chi2, p, dof, _ = stats.chi2_contingency(table)
    cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    shares = (table / table.sum()).round(4)
    rows.append(
        {
            "variable": column,
            "train_shares": dict(shares["train"]),
            "dev_shares": dict(shares["dev"]),
            "chi2": round(chi2, 2),
            "dof": dof,
            "p": round(p, 4),
            "cramers_V": round(cramers_v, 4),
        }
    )
display(pd.DataFrame(rows))

,variable,train_shares,dev_shares,chi2,dof,p,cramers_V
0,answer_type,"{'empty': 0.0077, 'number': 0.3942, 'other': 0...","{'empty': 0.0136, 'number': 0.4032, 'other': 0...",8.52,4,0.0744,0.0345
1,program_steps,"{'1': 0.5946, '2': 0.322, '3+': 0.0833}","{'1': 0.5923, '2': 0.325, '3+': 0.0827}",0.03,2,0.9837,0.0021
2,operation_family,"{'arithmetic': 0.9488, 'comparison': 0.0195, '...","{'arithmetic': 0.949, 'comparison': 0.0113, 't...",4.31,2,0.1160,0.0246
3,gold_facts,"{'1': 0.4559, '2+': 0.5441}","{'1': 0.4711, '2+': 0.5289}",0.66,1,0.4167,0.0096


## 3. Privacy scan of report text (all splits)

FinQA's reports are public earnings reports of S&P 500 companies, collected in FinTabNet (FinQA paper
§4.1). This scan looks for personal-data patterns in `pre_text`, `post_text` and table cells. It reads no
answers or programs. Matches are counted and shown masked; regular expressions cannot detect every
personal name, so the honorific count is only a rough proxy for references to named individuals.

In [4]:
PATTERNS = {
    "email": re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "us_phone": re.compile(r"\(?\b\d{3}\)?[ .-]\d{3}[ .-]\d{4}\b"),
    "ssn_like": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "url": re.compile(r"\b(?:https?://|www\.)\S+", re.I),
    "honorific_name": re.compile(r"\b(?:mr|mrs|ms|dr)\s?\.\s+[a-z]+", re.I),
}


def report_strings(record):
    yield from record["pre_text"]
    yield from record["post_text"]
    for row in record["table"]:
        yield from (str(cell) for cell in row)


def mask(text: str) -> str:
    return re.sub(r"[a-z0-9]", "x", text, flags=re.I)


counts, examples = Counter(), {}
for split, records in splits.items():
    for record in records:
        for text in report_strings(record):
            for name, pattern in PATTERNS.items():
                for match in pattern.finditer(text):
                    counts[(split, name)] += 1
                    examples.setdefault(name, (record["filename"], mask(match.group())))
scan = pd.DataFrame(
    [{"pattern": name, **{split: counts[(split, name)] for split in splits}} for name in PATTERNS]
)
display(scan)
print("first masked example per pattern (source report, masked match):")
for name, (filename, masked) in examples.items():
    print(f"  {name}: {filename} -> {masked}")

# Space-separated digit groups are usually adjacent table figures (FinQA tokenizes numbers with
# spaces); only hyphen/dot/parenthesis formats look like real phone numbers.
phone_styles, honorific_reports = Counter(), set()
for records in splits.values():
    for record in records:
        for text in report_strings(record):
            for match in PATTERNS["us_phone"].finditer(text):
                phone_styles[
                    "formatted (-, ., or parentheses)"
                    if re.search(r"[-.()]", match.group())
                    else "space-separated digits"
                ] += 1
            if PATTERNS["honorific_name"].search(text):
                honorific_reports.add(record["filename"])
print("us_phone matches by format:", dict(phone_styles))
print("distinct report pages with an honorific + name:", len(honorific_reports))

,pattern,train,dev,test
0,email,0,0,1
1,us_phone,31,0,4
2,ssn_like,0,0,0
3,url,114,32,12
4,honorific_name,14,0,0


first masked example per pattern (source report, masked match):
  url: ALLE/2016/page_29.pdf -> xxxx://xxx.xxx.xxx
  us_phone: INTC/2018/page_50.pdf -> xxx xxx xxxx
  honorific_name: ETR/2017/page_477.pdf -> xx . x
  email: APD/2018/page_30.pdf -> xxxxxxxxxxx@xxxxxxxxxx.xxx


us_phone matches by format: {'space-separated digits': 21, 'formatted (-, ., or parentheses)': 14}
distinct report pages with an honorific + name: 2


## 4. M3 result uncertainty (dev)

Percentile bootstrap 95% intervals (10,000 resamples of the 883 dev records, seed 42) for execution
accuracy, plus exact McNemar tests (two-sided binomial test on records where exactly one system is
correct) for paired comparisons on the same records.

In [5]:
dev = splits["dev"]
dev_by_id = {r["id"]: r for r in dev}


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]


arithmetic_rows = read_jsonl(PROCESSED_DIR / "conservative_arithmetic_dev_predictions.jsonl")
arithmetic_correct = {}
for row in arithmetic_rows:
    value = None
    if row["prediction"] is not None:
        value = official_str_to_num(str(row["prediction"]).replace("$", "").replace(" ", ""))
    arithmetic_correct[row["id"]] = compare_execution(
        value, dev_by_id[row["id"]]["qa"]["exe_ans"]
    ).is_correct
systems = {"notebook 08 conservative arithmetic": arithmetic_correct}
for condition in ("retrieved", "gold_facts_oracle"):
    rows = read_jsonl(PROCESSED_DIR / f"generator-t5-base-dev-v1_dev_{condition}_predictions.jsonl")
    systems[f"notebook 11 generator, {condition}"] = {
        r["record_id"]: r["execution_correct"] for r in rows
    }
order = [r["id"] for r in dev]
matrix = {
    name: np.array([values[i] for i in order], dtype=float) for name, values in systems.items()
}

rng = np.random.default_rng(SEED)
resamples = rng.integers(0, len(order), size=(10_000, len(order)))
interval_rows = []
for name, correct in matrix.items():
    boot = correct[resamples].mean(axis=1)
    interval_rows.append(
        {
            "system": name,
            "correct": int(correct.sum()),
            "records": len(correct),
            "execution_accuracy": round(correct.mean(), 4),
            "ci95_low": round(float(np.percentile(boot, 2.5)), 4),
            "ci95_high": round(float(np.percentile(boot, 97.5)), 4),
        }
    )
display(pd.DataFrame(interval_rows))


def mcnemar(a: np.ndarray, b: np.ndarray) -> dict:
    only_a, only_b = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    test = stats.binomtest(only_a, only_a + only_b, 0.5) if only_a + only_b else None
    return {
        "only_first_correct": only_a,
        "only_second_correct": only_b,
        "p_value": float(test.pvalue) if test else 1.0,
    }


names = list(matrix)
paired = [
    {"first": names[1], "second": names[0], **mcnemar(matrix[names[1]], matrix[names[0]])},
    {"first": names[2], "second": names[1], **mcnemar(matrix[names[2]], matrix[names[1]])},
]
display(pd.DataFrame(paired))

,system,correct,records,execution_accuracy,ci95_low,ci95_high
0,notebook 08 conservative arithmetic,35,883,0.0396,0.0272,0.0532
1,"notebook 11 generator, retrieved",523,883,0.5923,0.5606,0.6251
2,"notebook 11 generator, gold_facts_oracle",578,883,0.6546,0.6229,0.6852


,first,second,only_first_correct,only_second_correct,p_value
0,"notebook 11 generator, retrieved",notebook 08 conservative arithmetic,493,5,6.176173e-139
1,"notebook 11 generator, gold_facts_oracle","notebook 11 generator, retrieved",84,29,2.200773e-07


## Findings

Written from the executed outputs above; `docs/milestone-status.md` links each finding to its milestone deliverable.

**1. Input distributions are practically the same across splits.** Every KS statistic is at most 0.077, and medians are identical or differ by one unit. Question length, table size and year mentions show no detectable difference. Train and dev differ slightly in sentences per report (D = 0.075, p = 0.0003) and numeric-cell share (D = 0.072, p = 0.0007); train and test differ slightly in table rows (D = 0.077, p < 0.0001). These small but detectable shifts fit the paper's statement that the splits share no input reports, so they don't call for changing the official splits.

**2. Dev labels look like train labels.** On answer type, program length, operation family and number of gold facts, no train–dev difference is significant at 0.05 (smallest p = 0.074), and Cramér's V is at most 0.035. Dev is a representative model-selection split. No new statistic was computed on test labels.

**3. Privacy.** No SSN-like strings. One email address (test) and the formatted phone numbers are corporate shareholder-service contact details in the examples inspected. 21 of the 35 phone-pattern matches are space-separated table figures, i.e. false positives. The 158 URL matches are web addresses printed in the reports. Two report pages (14 matches) name individuals, such as executives, in public compensation or related-party disclosures. The corpus is public corporate disclosure with limited personal data about named executives and no evidence of private-individual records. Handling decisions are recorded in `docs/dataset-card.md`.

**4. M3 comparisons are statistically clear on dev.** Bootstrap 95% intervals: conservative arithmetic 0.0396 [0.0272, 0.0532]; generator with retrieved facts 0.5923 [0.5606, 0.6251]; gold-facts oracle 0.6546 [0.6229, 0.6852]. In paired McNemar tests, the generator beats arithmetic on 493 records and loses on 5 (p ≈ 6e-139). The oracle beats retrieved inputs on 84 records and loses on 29 (p ≈ 2e-7), so the retrieval gap is real, not noise.

**Limitations.** The intervals cover resampling of dev records only. They do not cover checkpoint selection on dev, seed variance or hardware variance. Regular expressions miss names written without honorifics, so the privacy scan is a screen, not a complete personal-data audit.